In [1]:
import pandas as pd
from datetime import datetime
from tqdm.auto import tqdm

# Import RAG

In [2]:
import sys
sys.path.append('../scripts')
import rag
import vectors

# Load synthetic questions

In [3]:
df_synth = pd.read_csv('../data/data-synth-question.csv', sep='\t', dtype=str)
df_synth

,pmid,ollama_seed,synthetic_question
0,40255413,0,Is gastrointestinal (GI) problems more common ...
1,40255413,1,Does autism spectrum disorders affect a child'...
2,40255413,2,What is one possible underlying factor contrib...
3,40255413,3,What is a common challenge that many individua...
4,40255413,4,How do gastrointestinal (GI) symptoms affect t...
...,...,...,...
495,40938164,0,Is autism a natural part of human diversity?
496,40938164,1,Can individuals with autism spectrum disorders...
497,40938164,2,What is it about individuals with autism spect...
498,40938164,3,What is the main difference between autism and...


# Demo RAG

In [4]:
# demonstrate RAG for one question
demo_query = df_synth.iloc[0]['synthetic_question']
print(demo_query)
print()
print(datetime.now())
demo_answer = rag.rag(demo_query, \
do_vector_search=False, num_results=2, model_handle_llm='llama3.2:1b', seed=42)
print(demo_answer)
print(datetime.now())

Is gastrointestinal (GI) problems more common in individuals with autism spectrum disorders than in the general population?

2025-09-14 00:57:28.227719
Based on the provided context and papers from PubMed, it appears that gastrointestinal (GI) problems are indeed more common in individuals with Autism Spectrum Disorders (ASD) compared to the general population.

The first paper by Eberly et al. (2025) from Frontiers in Neuroscience found that children with autism presented with more gastrointestinal symptoms at each time point, and they were also more likely to experience multiple and persistent GI symptoms throughout childhood.

Additionally, the longitudinal evaluation of gastrointestinal symptoms in children with autism spectrum disorder (Restrepo et al., 2025) showed that participants in this group reported more gastrointestinal symptoms without known etiology throughout childhood. The study also found associations between gastrointestinal symptoms and greater impairment in interna

# Generate answers by RAG

## Function

In [5]:
# common parameters for RAG
do_vector_search = False
print(do_vector_search)
num_results = 5
print(num_results)

False
5


In [6]:
def generate_answers(synth_records, model_handle_llm, seed):
    answers = []
    for record in tqdm(synth_records):
        query = record['synthetic_question']
        answer = rag.rag(query, \
        do_vector_search=rag.config['do_vector_search'], num_results=rag.config['num_results'], \
        model_handle_llm=model_handle_llm, seed=seed)
        pmid = record['pmid']
        ollama_seed_question = record['ollama_seed']
        answer_dict = {'pmid' : pmid, 'ollama_seed' : ollama_seed_question, \
        'answer_'+model_handle_llm : answer}
        answers.append(answer_dict)
    return pd.DataFrame.from_records(answers)

## Work

In [7]:
# sample size, dictionary format for dataset
sample_size = 30
synth_records = df_synth.to_dict('records')[:sample_size]
print(len(synth_records))

30


In [8]:
# generate answers for llama3.2
print(datetime.now())
answers_llama = generate_answers(synth_records=synth_records, \
model_handle_llm='llama3.2:1b', seed=42)
print(datetime.now())
answers_llama

2025-09-14 00:57:56.069935


  0%|          | 0/30 [00:00<?, ?it/s]

2025-09-14 01:27:09.212938


,pmid,ollama_seed,answer_llama3.2:1b
0,40255413,0,Based on the context provided by the papers fr...
1,40255413,1,Based on the provided context and papers from ...
2,40255413,2,Based on the context provided by the papers fr...
3,40255413,3,"Based on the provided papers from PubMed, a co..."
4,40255413,4,Based on the provided context and papers from ...
5,40264093,0,Based on the context provided by the papers yo...
6,40264093,1,Individuals on the autism spectrum can navigat...
7,40264093,2,Based on the provided context and papers from ...
8,40264093,3,Based on the context provided by the papers fr...
9,40264093,4,Individuals with Autism Spectrum Disorder (ASD...


In [9]:
# how long are the answers for llama3.2?
print(answers_llama['answer_llama3.2:1b'].apply(lambda x : \
len(x.split())).quantile([0.5, 0.75, 0.99, 1.0])) # percentiles

0.50    314.00
0.75    388.25
0.99    537.92
1.00    553.00
Name: answer_llama3.2:1b, dtype: float64


In [10]:
# generate answers for gemma3
print(datetime.now())
answers_gemma = generate_answers(synth_records=synth_records, \
model_handle_llm='gemma3:1b', seed=42)
print(datetime.now())
answers_gemma

2025-09-14 01:27:09.233673


  0%|          | 0/30 [00:00<?, ?it/s]

2025-09-14 02:01:18.869569


,pmid,ollama_seed,answer_gemma3:1b
0,40255413,0,"Based on the provided text, the answer is:\n\n..."
1,40255413,1,"Okay, here’s an answer based on the provided t..."
2,40255413,2,"Okay, based on the provided context and papers..."
3,40255413,3,"Okay, based on the provided context and papers..."
4,40255413,4,"Okay, let’s analyze the provided text and answ..."
5,40264093,0,"Okay, here’s an analysis of the provided text,..."
6,40264093,1,"Okay, let’s answer your question based on the ..."
7,40264093,2,"Okay, based on the provided context and the in..."
8,40264093,3,"Okay, here's an analysis of the provided text,..."
9,40264093,4,"Okay, here's an analysis of the provided paper..."


In [11]:
# how long are the answers for gemma3?
print(answers_gemma['answer_gemma3:1b'].apply(lambda x : \
len(x.split())).quantile([0.5, 0.75, 0.99, 1.0])) # percentiles

0.50     486.00
0.75     605.50
0.99     980.33
1.00    1016.00
Name: answer_gemma3:1b, dtype: float64


In [12]:
# put together answers from different models
df_synth_answer = answers_llama.merge(answers_gemma, on=['pmid', 'ollama_seed'], how='inner')
df_synth_answer

,pmid,ollama_seed,answer_llama3.2:1b,answer_gemma3:1b
0,40255413,0,Based on the context provided by the papers fr...,"Based on the provided text, the answer is:\n\n..."
1,40255413,1,Based on the provided context and papers from ...,"Okay, here’s an answer based on the provided t..."
2,40255413,2,Based on the context provided by the papers fr...,"Okay, based on the provided context and papers..."
3,40255413,3,"Based on the provided papers from PubMed, a co...","Okay, based on the provided context and papers..."
4,40255413,4,Based on the provided context and papers from ...,"Okay, let’s analyze the provided text and answ..."
5,40264093,0,Based on the context provided by the papers yo...,"Okay, here’s an analysis of the provided text,..."
6,40264093,1,Individuals on the autism spectrum can navigat...,"Okay, let’s answer your question based on the ..."
7,40264093,2,Based on the provided context and papers from ...,"Okay, based on the provided context and the in..."
8,40264093,3,Based on the context provided by the papers fr...,"Okay, here's an analysis of the provided text,..."
9,40264093,4,Individuals with Autism Spectrum Disorder (ASD...,"Okay, here's an analysis of the provided paper..."


# Write CSV file

In [13]:
# write CSV file
df_synth_answer.to_csv('../data/data-synth-answer.csv', index=False, sep='\t')

In [14]:
print(datetime.now())

2025-09-14 02:01:18.931528
